# GRIT_1 — Kaggle end-to-end workflow

Notebook này clone repo vào Kaggle, cài dependency, chuẩn bị riêng safety-task/preservation data, build projector và chạy vLLM + data-parallel training.

Trước khi chạy: chọn **GPU T4 x2**, bật **Internet**, sau đó thêm secret `HF_TOKEN` trong **Add-ons → Secrets**. Không ghi token trực tiếp vào notebook.

## 1. Clone hoặc cập nhật repository

In [2]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/namdeptraivcd/GRIT_1.git"
REPO_DIR = Path("/kaggle/working/GRIT_1")

if (REPO_DIR / ".git").exists():
    status = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain", "--untracked-files=no"],
        text=True,
    ).strip()
    if status:
        print("Repository has local changes; skipping git pull to preserve them.")
        print(status)
    else:
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
            check=True,
        )
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
print(f"Repository: {REPO_DIR}")
print(f"Commit: {commit}")

Cloning into '/kaggle/working/GRIT_1'...


Repository: /kaggle/working/GRIT_1
Commit: c7fbbab


## 2. Cài dependency

Nếu Kaggle yêu cầu restart session sau khi cài vLLM, hãy restart rồi chạy lại notebook từ đầu.

In [3]:
import shutil

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "-q",
        "-r",
        str(REPO_DIR / "requirements-vllm.txt"),
    ],
    check=True,
)
missing_commands = [name for name in ("vllm", "torchrun") if shutil.which(name) is None]
if missing_commands:
    raise RuntimeError(
        f"Installed packages, but command(s) are unavailable: {missing_commands}. "
        "Restart the Kaggle session, then rerun the notebook."
    )
print("Dependencies installed; vllm and torchrun are available.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.

Dependencies installed; vllm and torchrun are available.


## 3. Kiểm tra GPU và nạp Kaggle Secrets

In [4]:
import torch
from kaggle_secrets import UserSecretsClient

subprocess.run(["nvidia-smi", "-L"], check=True)
gpu_count = torch.cuda.device_count()
if gpu_count < 2:
    raise RuntimeError(
        f"GRIT_1 Kaggle workflow requires 2 GPUs, but found {gpu_count}. "
        "Select Accelerator: GPU T4 x2."
    )

secrets = UserSecretsClient()
try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Attach the HF_TOKEN secret to this Kaggle notebook before running."
    ) from exc

os.environ["HF_HOME"] = "/kaggle/working/hf-cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Cho flashinfer tìm thấy -lcuda mà không ghi vào thư mục hệ thống chỉ-đọc.
import glob
cuda_paths = sorted(glob.glob("/usr/**/libcuda.so.1", recursive=True))
if not cuda_paths:
    raise RuntimeError("Could not locate libcuda.so.1 on this Kaggle runtime.")
cuda_compat_dir = Path("/kaggle/working/cuda-compat")
cuda_compat_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(
    ["ln", "-sfn", cuda_paths[0], str(cuda_compat_dir / "libcuda.so")],
    check=True,
)
cuda_compat = str(cuda_compat_dir)
for variable in ("LD_LIBRARY_PATH", "LIBRARY_PATH"):
    current = os.environ.get(variable, "")
    os.environ[variable] = cuda_compat + (":" + current if current else "")
current_ldflags = os.environ.get("LDFLAGS", "")
os.environ["LDFLAGS"] = f"-L{cuda_compat}" + (" " + current_ldflags if current_ldflags else "")
print(f"CUDA compatibility link: {cuda_compat_dir / 'libcuda.so'} -> {cuda_paths[0]}")
print(f"CUDA link-time search path: {os.environ['LIBRARY_PATH']}")

print(f"CUDA ready: {gpu_count} GPUs; HF_TOKEN loaded without displaying it.")

GPU 0: Tesla T4 (UUID: GPU-0185d1ba-644a-e541-578e-81c0f0f9fc46)
GPU 1: Tesla T4 (UUID: GPU-67c9784e-e2f6-4cc3-7e58-30f08b2ae46a)
CUDA compatibility link: /kaggle/working/cuda-compat/libcuda.so -> /usr/local/cuda-12.8/compat/libcuda.so.1
CUDA link-time search path: /kaggle/working/cuda-compat:/usr/local/cuda/lib64/stubs
CUDA ready: 2 GPUs; HF_TOKEN loaded without displaying it.


## 4. Chọn chế độ và cấu hình run

Giữ `RUN_MODE = "smoke"` cho lần chạy đầu. Sau khi smoke test thành công, đổi thành `"full"`. Hai chế độ dùng data/artifact riêng; mỗi lần chạy có checkpoint và config riêng để không vô tình ghép optimizer cũ với model mới.

In [5]:
from datetime import datetime, timezone

import yaml

RUN_MODE = "smoke"  # "smoke" hoặc "full"
PROFILE = "small_0_6b"
LEARNING_RATE = 1.0e-6

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")

run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%fZ")
run_root = Path("/kaggle/working/grit1-runs") / RUN_MODE
config_path = run_root / "configs" / f"{PROFILE}-{run_id}.yaml"
config_path.parent.mkdir(parents=True, exist_ok=True)
with (REPO_DIR / "config.yaml").open(encoding="utf-8") as stream:
    config = yaml.safe_load(stream)
config_profiles = config["model_profiles"]
if PROFILE not in config_profiles:
    raise ValueError(f"Unknown PROFILE {PROFILE!r}; available={sorted(config_profiles)}")
config["update"]["learning_rate"] = LEARNING_RATE
with config_path.open("w", encoding="utf-8") as stream:
    yaml.safe_dump(config, stream, sort_keys=False, allow_unicode=True)

profile_config = config["model_profiles"][PROFILE]
checkpoint_dir = run_root / "checkpoints" / PROFILE / run_id
common = {
    "PYTHON_BIN": sys.executable,
    "PROFILE": PROFILE,
    "CONFIG": str(config_path),
    "DATA_ROOT": str(run_root / "data"),
    "PROJECTORS": str(run_root / "artifacts" / f"{PROFILE}_projectors.pt"),
    "MODEL_PATH": profile_config["policy_model"],
    "BASE_MODEL_PATH": profile_config["base_model"],
    "OUTPUT_DIR": str(checkpoint_dir),
    "INFERENCE_GPU": "0",
    "TRAIN_GPUS": "1",
    "NUM_TRAIN_GPUS": "1",
    "BATCH_SIZE": "1",
    "PRESERVATION_BATCH_SIZE": "1",
    "CENTRAL_FD_RADIUS": "0.05",
    "AUTO_PREPARE": "1",
}
mode_config = {
    "smoke": {
        "TASK_MAX_SAMPLES": "200",
        "VAL_SIZE": "50",
        "PRESERVE_MAX_SAMPLES": "60",
        "PROJECTOR_MAX_SAMPLES": "60",
        "PROJECTOR_MAX_LENGTH": "256",
        "EPOCHS": "1",
        "STEPS_PER_EPOCH": "2",
    },
    "full": {
        "TASK_MAX_SAMPLES": "10000",
        "VAL_SIZE": "1000",
        "PRESERVE_MAX_SAMPLES": "1000",
        "PROJECTOR_MAX_SAMPLES": "1000",
        "PROJECTOR_MAX_LENGTH": "1024",
        "EPOCHS": "2",
        "STEPS_PER_EPOCH": "10",
    },
}
os.environ.update(common)
os.environ.update(mode_config[RUN_MODE])

print(f"Mode: {RUN_MODE}")
print(f"Profile: {PROFILE}")
print(f"Run ID: {run_id}")
print(f"Run root: {run_root}")
print(f"Run config: {config_path}")
print(f"Checkpoint: {checkpoint_dir}")
print(f"Learning rate: {LEARNING_RATE}")

Mode: smoke
Profile: small_0_6b
Run ID: 20260929-152751-457426Z
Run root: /kaggle/working/grit1-runs/smoke
Run config: /kaggle/working/grit1-runs/smoke/configs/small_0_6b-20260929-152751-457426Z.yaml
Checkpoint: /kaggle/working/grit1-runs/smoke/checkpoints/small_0_6b/20260929-152751-457426Z
Learning rate: 1e-06


## 5. Chạy GRIT_1

GPU 0 chạy policy/safety vLLM; GPU 1 chạy training. Nếu launcher lỗi, cell sẽ in phần cuối của hai log vLLM.

In [6]:
launcher = REPO_DIR / "scripts" / "run_grit1_vllm_ddp.sh"
result = subprocess.run(
    ["bash", str(launcher)],
    cwd=REPO_DIR,
    env=os.environ.copy(),
)

if result.returncode != 0:
    for log_name in ("policy-vllm.log", "safety-vllm.log"):
        log_path = REPO_DIR / "logs" / log_name
        if log_path.exists():
            lines = log_path.read_text(errors="replace").splitlines()
            print(f"\n--- tail {log_name} ---")
            print("\n".join(lines[-100:]))
    result.check_returncode()

ready_marker = checkpoint_dir / "READY"
if not ready_marker.is_file():
    raise RuntimeError(f"Launcher exited successfully but did not create {ready_marker}")
print(f"Training complete. Checkpoint: {checkpoint_dir}")

[1/4] Loading task dataset: PKU-Alignment/PKU-SafeRLHF


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 1021.51ba/s]


[3/4] Writing safety-task parquet files to /kaggle/working/grit1-runs/smoke/data/safety_task
task_train=/kaggle/working/grit1-runs/smoke/data/safety_task/task_train.parquet rows=200
task_val=/kaggle/working/grit1-runs/smoke/data/safety_task/task_val.parquet rows=50
[4/4] Loading NSPO general-task preservation mixture


Generating train split: 20001 examples [00:00, 87065.19 examples/s]
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 1070.79ba/s]


preserve=/kaggle/working/grit1-runs/smoke/data/preservation/preserve_1000.parquet rows=60 sources=tatsu-lab/alpaca_farm,newfacade/LeetCodeDataset,openai/gsm8k domains={'code': 20, 'common_sense': 20, 'math': 20}
[1/5] Loading model: Qwen/Qwen3-0.6B


...

[4/5] Eigendecomposition: 100%|██████████| 84/84 [00:50<00:00,  1.67it/s]


[5/5] Saving projectors to /kaggle/working/grit1-runs/smoke/artifacts/small_0_6b_projectors.pt
Saved 84 projectors to /kaggle/working/grit1-runs/smoke/artifacts/small_0_6b_projectors.pt
model.layers.0.mlp.gate_proj: nullity=594 rank=430 sym=0.000e+00 idemp=3.609e-05
model.layers.0.mlp.up_proj: nullity=594 rank=430 sym=0.000e+00 idemp=3.609e-05
model.layers.0.mlp.down_proj: nullity=1500 rank=1572 sym=0.000e+00 idemp=7.285e-05
model.layers.1.mlp.gate_proj: nullity=822 rank=202 sym=0.000e+00 idemp=4.197e-05
model.layers.1.mlp.up_proj: nullity=822 rank=202 sym=0.000e+00 idemp=4.197e-05
model.layers.1.mlp.down_proj: nullity=1682 rank=1390 sym=0.000e+00 idemp=7.787e-05
model.layers.2.mlp.gate_proj: nullity=763 rank=261 sym=0.000e+00 idemp=4.117e-05
model.layers.2.mlp.up_proj: nullity=763 rank=261 sym=0.000e+00 idemp=4.117e-05
model.layers.2.mlp.down_proj: nullity=3071 rank=1 sym=0.000e+00 idemp=9.232e-05
model.layers.3.mlp.gate_proj: nullity=734 rank=290 sym=0.000e+00 idemp=4.034e-05
model.l

[W929 15:36:20.271270534 socket.cpp:209] [c10d] The hostname of the client socket cannot be retrieved. err=-3
`torch_dtype` is deprecated! Use `dtype` instead!
Generating train split: 200 examples [00:00, 18049.33 examples/s]
A new version of the following files was downloaded from https://huggingface.co/Qwen/Qwen3Guard-Stream-0.6B:
- configuration_qwen3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/Qwen/Qwen3Guard-Stream-0.6B:
- modeling_qwen3_guard.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/GRIT_1/grit1/train.py", line 542, in <mod


--- tail policy-vllm.log ---
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  31%|███▏      | 16/51 [00:00<00:01, 18.76it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  35%|███▌      | 18/51 [00:01<00:01, 18.94it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  39%|███▉      | 20/51 [00:01<00:01, 18.99it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  43%|████▎     | 22/51 [00:01<00:01, 18.69it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  47%|████▋     | 24/51 [00:01<00:01, 18.82it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  51%|█████     | 26/51 [00:01<00:01, 17.45it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  55%|█████▍    | 28/51 [00:01<00:01, 17.12it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  59%|█████▉    | 30/51 [00:01<00:01, 17.76it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  63%|██████▎   | 32/51 [00:01<00:01, 18.12it/s]
Capturing CUDA graphs (mi

CalledProcessError: Command '['bash', '/kaggle/working/GRIT_1/scripts/run_grit1_vllm_ddp.sh']' returned non-zero exit status 1.

## 6. Kiểm tra output

Sau khi hoàn tất, dùng **Save Version** để giữ checkpoint trong Kaggle Notebook Output. `/kaggle/working` có thể mất khi session kết thúc.

In [ ]:
for path in sorted(checkpoint_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(checkpoint_dir), f"{path.stat().st_size / 1024**2:.1f} MiB")